Nàdia Pelegay-Royo, Manon Vu Huu

[Original embedded image removed from public notebook]


# Machine Learning Final Project - Spotify Premium


Spotify is the biggest music streaming platform in the world.
Yet it is met with a fair share of controversy.

Indeed it has often in the recent years been accused of planting songs to users' experiences for commercial profit. It's recommendation system, rather than democratizing music discovery, often seems to reproduce the industry's inequalities, favoring already established artists and labels. [1]

We were tired of getting recommended the same commercial songs and believe music is an art form that has to be explored with less bias. (you can remove this Nadia:) Moreover, in good 21st century modern women spirit we will not accept to be told what to listen to, we want to feel free of our choices.

This study thus aims to build a "better" spotify song recommendation system, that's purely based on sound and not the industry's intricate political dynamics.
Our work's title "Spotify Premium" ironically stems from this.


# 1. Imports

Below are all the libraries we will be using for this project.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import kagglehub
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier



In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth, SpotifyClientCredentials
import spotipy.util as util


# 2. Data Work

The Spotify Web API is a platform provided directly by Spotify that allows independent developers to access Spotify's catalog and user data.

Data extracted from the source seemed like a goldmine for us, so we tried to access it like shown below:

In [ ]:
# OPTIONAL: Spotify API exploration. Requires your own Spotify app credentials.
# See README.md and .env.example. This cell never embeds credentials.
import os
from dotenv import load_dotenv
load_dotenv()

sp = None
user = None
if all(os.getenv(k) for k in ("SPOTIPY_CLIENT_ID", "SPOTIPY_CLIENT_SECRET", "SPOTIPY_REDIRECT_URI")):
    from spotipy import Spotify
    from spotipy.oauth2 import SpotifyOAuth
    sp = Spotify(auth_manager=SpotifyOAuth(scope="playlist-modify-private", cache_path=".spotify_cache"))
    user = sp.current_user()
else:
    print("Optional API exploration skipped. See README.md for Spotify credentials setup.")


In [ ]:
# Optional API example; run the Spotify setup cell with your own credentials first.
if sp is not None:
    #GETTING ARTIST INFO

    artist_id = '35l9BRT7MXmM8bv2WDQiyB'

    artist_info = sp.artist(artist_id)
    print(f"\nArtist Info:")
    print(f"Name: {artist_info['name']}")
    print(f"Genres: {', '.join(artist_info['genres'])}")
    print(f"Followers: {artist_info['followers']['total']}")
    print(f"Popularity: {artist_info['popularity']}")
    print(f"Spotify URL: {artist_info['external_urls']['spotify']}")

In [ ]:
# Optional API example; run the Spotify setup cell with your own credentials first.
if sp is not None:
    #GETTING TRACK METADATA

    track_id = '5sA3i6SGkILMiZW6Fhx4wv'

    track_info = sp.track(track_id)
    print(f"\nTrack Metadata:")
    print(f"Name: {track_info['name']}")
    print(f"Artists: {', '.join([artist['name'] for artist in track_info['artists']])}")
    print(f"Album: {track_info['album']['name']}")
    print(f"Release Date: {track_info['album']['release_date']}")
    print(f"Duration (ms): {track_info['duration_ms']}")
    print(f"Explicit: {track_info['explicit']}")
    print(f"Popularity: {track_info['popularity']}")
    print(f"Spotify URL: {track_info['external_urls']['spotify']}")

In [ ]:
# Optional API example; run the Spotify setup cell with your own credentials first.
if sp is not None:
    #GETTING TRACK AUDIO FEATURES

    track_id = '5sA3i6SGkILMiZW6Fhx4wv'

    features = sp.audio_features([track_id])[0]
    if features:
        print(f"\nAudio Features:")
        print(f"Danceability: {features['danceability']}")
        print(f"Energy: {features['energy']}")
        print(f"Key: {features['key']}")
        print(f"Loudness: {features['loudness']} dB")
        print(f"Mode: {'Major' if features['mode'] == 1 else 'Minor'}")
        print(f"Speechiness: {features['speechiness']}")
        print(f"Acousticness: {features['acousticness']}")
        print(f"Instrumentalness: {features['instrumentalness']}")
        print(f"Liveness: {features['liveness']}")
        print(f"Valence: {features['valence']}")
        print(f"Tempo: {features['tempo']} BPM")
        print(f"Time Signature: {features['time_signature']}")
    else:
        print("No audio features found for this track.")

This would've been all we needed to start working on a model, but unfortunately as of November 2024 Spotify had discontinued access to its Audio Features API.

Gutted, we decided to resort to a kaggle dataset [2].
Amongst a lot of datasets that made audio features available, we chose this one for its extensive data (around 90 000 songs featured, which is more than enough for the scope of our research), the features it offered, as well as the readability and simpleness of its documentation.

In [ ]:
#ACCESSING THE DATA

path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset")

df = pd.read_csv(path+'/dataset.csv')

print(df.shape)
print(df.describe())

In [ ]:
#CURATING THE DATASET

df = df.drop_duplicates(subset=['track_id'])
df.drop(df[df.isnull().any(axis=1)].index, inplace=True)
df = df.reset_index(drop=True)


print(df.shape)
print(df.describe())

In [ ]:
#CHECKING OUT OUR VALUES


nc = df[df.columns[(df.dtypes == 'float64') | (df.dtypes == 'int64')]]
nc.hist(figsize=(16,12), bins=30, xlabelsize=8, ylabelsize=8)
plt.tight_layout()
plt.show()

From the above histograms, we get that:

*   danceability, energy, and valence have normal or right-skewed distributions, so most tracks have moderate values.

*   speechiness, instrumentalness, and liveness are more skewed toward zero, so most tracks have low values.

*   key and mode are categorical

*   tempo has a more or less Gaussian distribution, centered around typical BPM values.

These will be important to keep in mind when customizing our recommendations.

# 3. Introducing Comparative Analysis

Before we start making recommendation systems, we define a comparative analysis metric to be able to compare how good they are amongst themselves and against Spotify's actual recommendation system.

For this we will use cosine similarity, because it is a scale-invariant measure. Indeed we will be using normalized data so this makes sense for us.

In [ ]:
def cos_similarity(og, comp):
    og_index_list = df[df['track_id'] == og].index.tolist()
    comp_index_list = df[df['track_id'] == comp].index.tolist()

    if not og_index_list or not comp_index_list:
        return 0

    og_index = og_index_list[0]
    comp_index = comp_index_list[0]

    og_vector = normalized_features[og_index].reshape(1, -1)
    comp_vector = normalized_features[comp_index].reshape(1, -1)

    similarity = cosine_similarity(og_vector, comp_vector)[0][0]
    return similarity


For reference, we will put out random recommendations from the dataset and check their cosine similarity:

In [ ]:
def random_recommendations(n=10, exclude_track_id=None):

    temp_df = df.copy()
    if exclude_track_id:
        temp_df = temp_df[temp_df['track_id'] != exclude_track_id]
    return temp_df.sample(n=n)[['track_id', 'track_name', 'artists', 'album_name', 'track_genre']]


In [ ]:
og = '2eAvDnpXP5W0cVtiI0PUxV'
random_rec = random_recommendations(n=20, exclude_track_id=og)

similarities = []
for rec in random_rec['track_id']:
    similarities.append(cos_similarity(og, rec))

x = list(range(1, len(similarities)+1))
avg = sum(similarities) / len(similarities)

plt.figure(figsize=(10, 6))
plt.plot(x, similarities, marker='o', color='skyblue')
plt.axhline(y=avg, color='red', linestyle='--', label=f'Average: {avg:.2f}')
plt.xticks(x)
plt.xlabel('Song Number')
plt.ylabel('Cosine Similarity')
plt.title('Cosine Similarity of Random Tracks Compared to Original Track')
plt.legend()
plt.tight_layout()
plt.show()

And also for reference, here are recommendations that Spotify made for the same track and their cosine similarity:

In [ ]:
og = '2eAvDnpXP5W0cVtiI0PUxV'
spotify_rec = ['2nMeu6UenVvwUktBCpLMK9', '4Y8KhqNtOMToFdUOAJpECt', '7D0RhFcb3CrfPuTJ0obrod',
 '41zXlQxzTi6cGAjpOXyLYH', '3dYD57lRAUcMHufyqn9GcI', '2mdEsXPu8ZmkHRRtAdC09e',
 '4Dvkj6JhhA12EX05fT7y2e', '2mlNgAeIBnL78ZriXgrRHz', '0u2P5u6lvoDfwTYjAADbn4',
 '3AJwUDP919kvQ9QcozQPxg', '7D0RhFcb3CrfPuTJ0obrod', '1oAwsWBovWRIp7qLMGPIet',
 '3vkCueOmm7xQDoJ17W1Pm3', '2vjX1VkntJPIQkqwbJQvoa', '6dOtVTDdiauQNBQEDOtlAB',
 '3ZCTVFBt2Brf31RLEnCkWJ', '4HBZA5flZLE435QTztThqH', '1Dr5JexwA15wmKe7Y7maA9',
 '1SKPmfSYaPsETbRHaiA18G', '5cF0dROlMOK5uNZtivgu50']



similarities = []
for rec in spotify_rec:
    similarities.append(cos_similarity(og, rec))

x = list(range(1, len(similarities)+1))
avg = sum(similarities) / len(similarities)

plt.figure(figsize=(10, 6))
plt.plot(x, similarities, marker='o', color='skyblue')
plt.axhline(y=avg, color='red', linestyle='--', label=f'Average: {avg:.2f}')
plt.xticks(x)
plt.xlabel('Song Number')
plt.ylabel('Cosine Similarity')
plt.title('Cosine Similarity of Spotify Tracks Compared to Original Track')
plt.legend()
plt.tight_layout()
plt.show()

For this graph we put songs that are currently on Spotify, but they're not necessarily in the dataset (it's 3 years old), hence some points being at 0 and putting the average down.
Nevertheless we can observe that a lot of recommendations have low cosine similarity which suggets the songs are not actually similar to the original song.

# 4. Our 1st Recommendation System - kNN

Now that we have data to build with, we can start recommending songs.

We will start by doing this instinctively through a kNN model because what we're looking for here is similarity.

We choose the following features for their musical relevance and low collinearity.

In [ ]:
features = ['danceability', 'energy', 'valence', 'tempo', 'acousticness', 'liveness', 'speechiness', 'instrumentalness']

We normalize the features to ensure all of them contribute equally to similarity computations.

In [ ]:

normalized_features = MinMaxScaler().fit_transform(df[features])


In [ ]:
#KNN


knn_model = NearestNeighbors(n_neighbors=5)
knn_model.fit(normalized_features)


def recommend_songs(track_id, nb=20):
    song_index = df.index[df['track_id'] == track_id].tolist()[0]
    distances, indices = knn_model.kneighbors([normalized_features[song_index]], n_neighbors=nb+1)
    recommendations = df.iloc[indices[0][1:]]
    return recommendations[['track_id', 'track_name', 'artists', 'album_name', 'track_genre']]

sample_track_id = '2eAvDnpXP5W0cVtiI0PUxV'
sample_row = df[df['track_id'] == sample_track_id]
print(sample_row.iloc[0][['track_name', 'artists']])
print()
recommended_songs = recommend_songs(sample_track_id)
print(recommended_songs)
print()
print(recommended_songs['track_id'])

This is already great and fulfills our minimum requirements for this project.

So we can compute the cosine similarity for it.

In [ ]:
og = '2eAvDnpXP5W0cVtiI0PUxV'

similarities = []
for rec in recommended_songs['track_id']:
    similarities.append(cos_similarity(og, rec))

x = list(range(1, len(similarities)+1))
avg = sum(similarities) / len(similarities)

plt.figure(figsize=(10, 6))
plt.plot(x, similarities, marker='o', color='skyblue')
plt.axhline(y=avg, color='red', linestyle='--', label=f'Average: {avg:.2f}')
plt.xticks(x)
plt.xlabel('Song Number')
plt.ylabel('Cosine Similarity')
plt.title('Cosine Similarity of KNN Recommendation Tracks Compared to Original Track')
plt.legend()
plt.tight_layout()
plt.show()



What we could do for an even better listener experience is to have customizable recommendations:

In [ ]:
#CUSTOMIZED KNN

def recommend_songs_custom(track_id, num_recommendations=5, filters=None, emphasize=None):

    song_index = df.index[df['track_id'] == track_id].tolist()[0]

    #filter df

    filtered_df = df.copy()
    if filters:
        if 'track_genre' in filters:
            filtered_df = filtered_df[filtered_df['track_genre'] == filters['track_genre']]
        if 'popularity_min' in filters:
            filtered_df = filtered_df[filtered_df['popularity'] >= filters['popularity_min']]
        if 'artist' in filters:
            filtered_df = filtered_df[filtered_df['artists'].str.contains(filters['artist'], case=False)]

    if filtered_df.empty:
        print("No songs match the filter criteria.")
        return pd.DataFrame()

    #normalize filtered features and refit knn model on filtered data
    filtered_normalized = MinMaxScaler().fit_transform(filtered_df[features])
    knn = NearestNeighbors(n_neighbors=min(num_recommendations + 1, len(filtered_df)))
    knn.fit(filtered_normalized)

    # query vector from original song
    query_vector = normalized_features[song_index].copy()
    if emphasize:
        for feat, weight in emphasize.items():
            if feat in features:
                idx = features.index(feat)
                query_vector[idx] *= weight

    #knn
    distances, indices = knn.kneighbors([query_vector])
    recommendations = filtered_df.iloc[indices[0]]
    input_track_id = df.iloc[song_index]['track_id']
    recommendations = recommendations[recommendations['track_id'] != input_track_id]

    result_df = recommendations.head(num_recommendations)[['track_id', 'track_name', 'artists', 'album_name', 'track_genre']]
    track_ids = result_df['track_id'].tolist()

    return result_df, track_ids



#example
recommended = recommend_songs_custom(
    track_id='2eAvDnpXP5W0cVtiI0PUxV',
    num_recommendations=10,
    filters={'track_genre': 'pop', 'popularity_min': 70},
    emphasize={'danceability': 1.5, 'valence': 1.2}
)

print(df.iloc[21054][['track_name', 'artists']])
print()
print(recommended[0])
print()
print(recommended[1])



What's more, we can make this an interactive user experience:

In [ ]:
def interactive_recommendation():
    print("\nWelcome to Nàdia and Manon's song recommender")

    track_id = input("Enter a track ID to base recommendations on: ").strip()

    #filter
    filters = {}
    apply_filters = input("Do you want to apply any filters? (y/n): ").strip().lower()
    if apply_filters == 'y':

        print("\nAvailable genres:")
        unique_genres = sorted(df['track_genre'].dropna().unique())
        print(", ".join(unique_genres))

        genre = input("Filter by genre (leave blank to skip): ").strip()
        if genre:
            filters['track_genre'] = genre
        try:
            pop_min = int(input("Minimum popularity (0-100, leave blank to skip): ").strip())
            filters['popularity_min'] = pop_min
        except ValueError:
            pass
        artist_filter = input("Filter by artist (leave blank to skip): ").strip()
        if artist_filter:
            filters['artist'] = artist_filter

    #feature emphasis
    emphasize = {}
    print("\nSet feature emphasis (scale: 1 = neutral, >1 = prioritize, <1 = de-prioritize)")
    for feat in features:
        try:
            weight = float(input(f"{feat} (0-2, default 1): ").strip() or 1.0)
            emphasize[feat] = weight
        except ValueError:
            emphasize[feat] = 1.0

    #number of recos
    try:
        num_recommendations = int(input("Number of recommendations: ").strip())
    except ValueError:
        num_recommendations = 5

    #recos using previous function
    result_df, track_ids = recommend_songs_custom(track_id, num_recommendations, filters,emphasize)

    if not result_df.empty:
        print("\nRecommended Songs:\n")
        print(result_df)
    else:
        print("\nNo recommendations found. Try changing filters or base song.")

    return result_df, track_ids


recommended = interactive_recommendation()

# 5. Our 2nd Recommendation System - kNN but different

Another recommendation we wanted to implement with kNN is a more "flow" based one.

In fact, we have realized from personal experience that sometimes we want a playlist with a nice flow and songs that feel nicely connected rather than one that matches the original song perfectly. That is especially the case for longer listening parties.

The following implements a similar kNN model to before, but with this implementation in mind.

In [ ]:
def recommendation_flow(og_track_id, nb=10):

    visited = set([og_track_id])
    flow = []

    og_index = df.index[df['track_id'] == og_track_id].tolist()[0]
    now = normalized_features[og_index].reshape(1, -1)

    for _ in range(nb):
        #exclude already visited
        candidate_mask = df['track_id'].apply(lambda tid: tid not in visited)
        candidate_df = df[candidate_mask].reset_index(drop=True)
        candidate_vectors = MinMaxScaler().fit_transform(candidate_df[features])

        #knn
        knn = NearestNeighbors(n_neighbors=1)
        knn.fit(candidate_vectors)
        distance, index = knn.kneighbors(now)

        #get the next track
        next_row = candidate_df.iloc[index[0][0]]
        next_track_id = next_row['track_id']
        flow.append(next_row)
        visited.add(next_track_id)

        #update current vector
        now = candidate_vectors[index[0][0]].reshape(1, -1)

    return pd.DataFrame(flow)[['track_id', 'track_name', 'artists', 'album_name', 'track_genre']]

flow_df = recommendation_flow('2eAvDnpXP5W0cVtiI0PUxV', nb=10)
print(flow_df)

For this recommendation system, we will compare the cosine similarity of every i-th song to the i+1-th song, and compare it to the cosine similarity of the first and last song. This enables us to prove our "flow" effect.


In [ ]:
flow_df = recommendation_flow('2eAvDnpXP5W0cVtiI0PUxV', nb=10)
track_ids = flow_df['track_id'].tolist()

pairwise_similarities = [cos_similarity(track_ids[i], track_ids[i + 1]) for i in range(len(track_ids)-1)]

eg_end_similarity = cos_similarity(track_ids[0], track_ids[-1])

x = list(range(1, len(pairwise_similarities) + 1))
plt.figure(figsize=(10, 6))
plt.plot(x, pairwise_similarities, marker='o', color='skyblue', label='Consecutive Track Similarities')
plt.axhline(y=eg_end_similarity, color='red', linestyle='--', label=f'First-to-Last Similarity: {eg_end_similarity:.2f}')
plt.xticks(x)
plt.xlabel('Step in Flow (i → i+1)')
plt.ylabel('Cosine Similarity')
plt.title('Cosine Similarity Between Consecutive Tracks in Flow')
plt.legend()
plt.tight_layout()
plt.show()

# 6. Our 3rd recommendation system - Logistic Regression


Another model we had learned about early on in class was logistic regression. It is still a classification algorithm but it works differently than the kNN by the fact that it does some learning and is parametric.

We wanted to see how this one would perform in a recommendation setting.

In order to make this work, we start by computing the cosine similarity between the original song and all the other tracks in the dataset. Above a certain threshold we label the tracks as similar (1) and not similar (0). Using these labels and the normalized audio features, we train a logistic regression model that estimates the likelihood of a song's similarity to the original. The model then predicts similarity probabilities for all tracks and recommends the top n.

In [ ]:
og = '2eAvDnpXP5W0cVtiI0PUxV'
threshold = 0.95

# Get index and vector for original song
og_index = df[df['track_id'] == og].index[0]
og_vector = normalized_features[og_index].reshape(1, -1)

# Compute cosine similarities for all tracks
all_similarities = cosine_similarity(normalized_features, og_vector).flatten()

labels = (all_similarities >= threshold).astype(int)

# logistic regression
X = normalized_features
y = labels
clf = LogisticRegression()
clf.fit(X, y)

# probability of being similar
probabilities = clf.predict_proba(X)[:, 1]

# top n recommendations
n = 20
df['similarity_score'] = probabilities
recommendations = (
    df[df['track_id'] != og]
    .drop_duplicates(subset=['track_name', 'artists'])
    .sort_values(by='similarity_score', ascending=False)
    .head(n)
)

recommendations[['track_name', 'artists', 'similarity_score']]




In [ ]:
og = '2eAvDnpXP5W0cVtiI0PUxV'

similarities = []
for rec in recommendations['track_id']:
    similarities.append(cos_similarity(og, rec))

x = list(range(1, len(similarities)+1))
avg = sum(similarities) / len(similarities)

plt.figure(figsize=(10, 6))
plt.plot(x, similarities, marker='o', color='skyblue')
plt.axhline(y=avg, color='red', linestyle='--', label=f'Average: {avg:.2f}')
plt.xticks(x)
plt.xlabel('Song Number')
plt.ylabel('Cosine Similarity')
plt.title('Cosine Similarity of Logistic Regression Tracks Compared to Original Song')
plt.legend()
plt.tight_layout()
plt.show()


# 7. Our 4th recommendation system - Random Forest Classifier

Further down the course we also learned about random forest classifier.
The method here is very similar to logistic regression. The only advantage here is that we get more complex patters in the audio features.

In [ ]:
og = '2eAvDnpXP5W0cVtiI0PUxV'
threshold = 0.95

#Get index and vector for original song
og_index = df[df['track_id'] == og].index[0]
og_vector = normalized_features[og_index].reshape(1, -1)

#Compute cosine similarities for all tracks
all_similarities = cosine_similarity(normalized_features, og_vector).flatten()

labels = (all_similarities >= threshold).astype(int)

# Random Forest
X = normalized_features
y = labels
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X, y)

# probability of being similar
probabilities = clf.predict_proba(X)[:, 1]

# top n recommendations
n = 20
df['similarity_score'] = probabilities

recommendations = (
    df[df['track_id'] != og]
    .drop_duplicates(subset=['track_name', 'artists'])
    .sort_values(by='similarity_score', ascending=False)
    .head(n)
)

recommendations[['track_name', 'artists', 'similarity_score']]




In [ ]:
og = '2eAvDnpXP5W0cVtiI0PUxV'

similarities = []
for rec in recommendations['track_id']:
    similarities.append(cos_similarity(og, rec))

x = list(range(1, len(similarities)+1))
avg = sum(similarities) / len(similarities)

plt.figure(figsize=(10, 6))
plt.plot(x, similarities, marker='o', color='skyblue')
plt.axhline(y=avg, color='red', linestyle='--', label=f'Average: {avg:.2f}')
plt.xticks(x)
plt.xlabel('Song Number')
plt.ylabel('Cosine Similarity')
plt.title('Cosine Similarity of Random Forest Tracks Compared to Original Song')
plt.legend()
plt.tight_layout()
plt.show()

# 8. Bonus: Playlist Making

As to follow through with our mission, and build something that we can actually use in our daily life, we have made it possible to connect the Spotify Web API to our own accounts to add our recommendations to it.

### Optional playlist export

Once you have a recommendation list, save its `track_id` values as a text file (one ID per line) and run `python -m src.playlist_creation --tracks tracks.txt --name "My Recommendations"`. This replaces the original credential-dependent cell.


# 9. Limitations & Discussion

The first obvious limitations lay in the dataset we're using:

*   It was made 3 years ago so it does not include more recent songs and needs updates in terms of popularity data.

*   We are only using a handful of audio features, there's always more we could use: keys, lyrics, information such as if the user replays the song or skips it often, language, etc. We are working with a minimum here.


Another important point of limitation discussion is that sometimes users do want to listen to what everyone's listening to: relatability and conformity are non negligeable aspects of our taste. Spotify imposes it to us, but in an ideal world we could choose how much of that we want. Our study tries to exclude it completely by incorporating a solely technical aspect of music.





# 10. References

[1] Tófalvy, T. and Koltai, J., 2023. “Splendid Isolation”: The reproduction of music industry inequalities in Spotify’s recommendation system. New Media & Society.

[2] Pandya, M. (n.d.). Spotify Tracks Dataset. Kaggle. Available at: https://www.kaggle.com/datasets/maharshipandya/spotify-tracks-dataset